# Experimentos de modelado

Este cuaderno resume los resultados generados por `src/models/run_experiments.py`. Partimos de los artefactos guardados (métricas, resumen, predicciones y modelos) para:

1. Visualizar y comparar las métricas por modelo/fold.
2. Analizar los errores en el conjunto de test (residuos y dispersión y_true vs. y_pred).
3. Revisar la importancia de características (cuando el modelo lo permite) y documentar conclusiones.

### 1. Preparación
Actualiza la ruta `EXPERIMENT_DIR` con la carpeta específica (p. ej., `experiment_YYYYMMDD_HHMMSS`).


In [7]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

EXPERIMENT_BASE_DIR = Path("../data/results/modeling/experiments")

runner_experiments = sorted(
    [d for d in EXPERIMENT_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("runner_id_")],
    reverse=True,
)
if not runner_experiments:
    raise FileNotFoundError("Runner ID experiment directories not found.")

selected_dir = runner_experiments[0]
print(f"Using experiment directory: {selected_dir}")

metrics_path = selected_dir / "metrics.csv"
summary_path = selected_dir / "summary.csv"
predictions_path = selected_dir / "predictions.parquet"
feature_cols_path = selected_dir / "feature_columns.json"

if not metrics_path.exists() or not summary_path.exists():
    raise FileNotFoundError("Required metrics.csv or summary.csv not found in the selected experiment directory.")

experiment_data = {
    "dir": selected_dir,
    "metrics": pd.read_csv(metrics_path),
    "summary": pd.read_csv(summary_path),
    "target_label": "fatigue_score",
}

if predictions_path.exists():
    experiment_data["pred"] = pd.read_parquet(predictions_path)
else:
    print("predictions.parquet not found; setting predictions to None.")
    experiment_data["pred"] = None

if feature_cols_path.exists():
    experiment_data["feature_columns"] = json.loads(feature_cols_path.read_text())
else:
    experiment_data["feature_columns"] = None

Using experiment directory: ../data/results/modeling/experiments/runner_id_20251217_190729


In [8]:
metrics_table = (
    experiment_data["summary"]
    .pivot_table(
        index="model",
        columns="split",
        values=["mae_mean", "rmse_mean", "r2_mean"],
    )
    .round(4)
)

display(metrics_table)


mae_mean         r2_mean         rmse_mean        
split                        cv    test      cv    test        cv    test
model                                                                    
catboost                 0.0613  0.0494  0.5849  0.7907    0.0819  0.0640
elasticnet               0.0601  0.0597  0.6071  0.6639    0.0805  0.0811
gradient_boosting        0.0592  0.0477  0.6166  0.8033    0.0781  0.0620
hist_gradient_boosting   0.0604  0.0487  0.5863  0.7898    0.0815  0.0641
random_forest            0.0603  0.0532  0.5866  0.7420    0.0811  0.0710
xgboost                  0.0601  0.0496  0.5989  0.7899    0.0803  0.0641

### 2. Comparativa de métricas
Gráficas para comparar MAE/RMSE/R² por modelo y partición.


In [9]:
summary_df = experiment_data["summary"].copy()
label = experiment_data["target_label"]
print(f"Objetivo: {label} (Experimento: {experiment_data['dir'].name})")

fig_mae = px.bar(
    summary_df,
    x="model",
    y="mae_mean",
    color="split",
    error_y="mae_std",
    title=f"MAE por modelo y partición ({label})",
    labels={"model": "Modelo", "split": "Partición", "mae_mean": "MAE"},
    text_auto=".4f"   
)
fig_mae.update_traces(textposition="outside", cliponaxis=False)
fig_mae.show()

fig_rmse = px.bar(
    summary_df,
    x="model",
    y="rmse_mean",
    color="split",
    error_y="rmse_std",
    title=f"RMSE por modelo y partición ({label})",
    labels={"model": "Modelo", "split": "Partición", "rmse_mean": "RMSE"},
    text_auto=".4f"
)
fig_rmse.update_traces(textposition="outside", cliponaxis=False)
fig_rmse.show()

fig_r2 = px.bar(
    summary_df,
    x="model",
    y="r2_mean",
    color="split",
    error_y="r2_std",
    title=f"R² por modelo y partición ({label})",
    labels={"model": "Modelo", "split": "Partición", "r2_mean": "R²"},
    text_auto=".4f"
)
fig_r2.update_traces(textposition="outside", cliponaxis=False)
fig_r2.show()


Objetivo: fatigue_score (Experimento: runner_id_20251217_190729)


### 3. Residuos y dispersiones (test)

Inspeccionamos cómo se comporta cada modelo sobre el conjunto de test.


In [10]:
pred_df = experiment_data["pred"].copy()
label = experiment_data["target_label"]
pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

fig_scatter = px.scatter(
    pred_df,
    x="y_true",
    y="y_pred",
    color="model",
    title=f"Valor real vs. predicho ({label}, test)",
    labels={"y_true": f"{label} (real)", "y_pred": f"{label} (predicho)", "model": "Modelo"}
)
fig_scatter.add_trace(
    go.Scatter(
        x=[pred_df.y_true.min(), pred_df.y_true.max()],
        y=[pred_df.y_true.min(), pred_df.y_true.max()],
        mode="lines",
        name="Ideal",
        line=dict(dash="dash", color="gray")
    )
)
fig_scatter.show()

fig_res = px.box(
    pred_df,
    x="model",
    y="residual",
    title=f"Distribución de residuos ({label}, test)",
    labels={"model": "Modelo", "residual": "Residuo"}
)
fig_res.update_traces(boxmean=True)
fig_res.show()



### 4. Importancia de características

Analizamos la contribución de las 20 características más relevantes para cada modelo que expone importancias o coeficientes, a fin de documentar qué señales fisiológicas o biomecánicas dominan las predicciones.

In [11]:
import joblib
import numpy as np

feature_columns = experiment_data["feature_columns"]
label = experiment_data["target_label"]
available_models = sorted(experiment_data["summary"]["model"].unique())
print(f"Objetivo: {label} (experimento: {experiment_data['dir'].name})")

for model_name in available_models:
    model_path = experiment_data["dir"] / f"{model_name}_best.joblib"
    if not model_path.exists():
        print(f"Artefacto del modelo no encontrado para {model_name}.")
        continue

    pipeline = joblib.load(model_path)
    model = pipeline.named_steps["model"]
    importances = getattr(model, "feature_importances_", None)

    if importances is None:
        coef = getattr(model, "coef_", None)
        if coef is not None:
            importances = np.abs(np.ravel(coef))
        else:
            print(f"  {model_name} no expone importancias ni coeficientes; se omite.")
            continue

    feature_names = (
        feature_columns
        if feature_columns is not None and len(feature_columns) == len(importances)
        else [f"feature_{i}" for i in range(len(importances))]
    )

    fi = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )

    fig = px.bar(
        fi,
        x="feature",
        y="importance",
        title=f"20 características más relevantes ({model_name}, {label})",
        labels={"feature": "Característica", "importance": "Importancia"}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


Objetivo: fatigue_score (experimento: runner_id_20251217_190729)


  hist_gradient_boosting no expone importancias ni coeficientes; se omite.
